# 임베딩 모델 비교: bge-m3 · KURE-v1 · CLOVA Studio

실전 2 연구계획서 E2(검색 방식·모델 비교) 사전 점검. **로컬 Mac에서만 실행**합니다 (카탈로그 데이터가 밖으로 나가지 않음).

| 모델 | 실행 위치 | 비고 |
|---|---|---|
| bge-m3 (`BAAI/bge-m3`) | 이 노트북 (로컬, Apple GPU) | 처음 실행 시 모델 약 2GB 내려받음 |
| KURE-v1 (`nlpai-lab/KURE-v1`) | 이 노트북 (로컬) | bge-m3를 한국어로 추가 학습한 모델 |
| CLOVA Studio v2 | API (`app/services/embedding.py`) | 분당 60건 제한 → 표 2,364개에 약 40분. `RUN_CLOVA = True`일 때만 |
| 키워드 (현재 검색) | `app/services/search.py` | 비교 기준 |

**실행 방법** (프로젝트 루트에서. 프로젝트 의존성·Docker 이미지는 바꾸지 않음)
```bash
uv run --with jupyterlab --with sentence-transformers jupyter lab notebooks/embedding_compare.ipynb
```

**평가 방식:** 일상어 질문 10개(`scripts/check_embedding.py`의 `PROBES`)로 전체 표를 검색해, 규칙으로 정한 관련 표(표명에 규칙 단어가 있는 표)가 Top-10에 몇 개 드는지, 첫 관련 표 순위를 봅니다. **골든셋이 아닌 규칙 기반 예비 점검**이라 정확도 주장에는 쓰지 않고, 정식 비교는 골든셋으로 하는 E2에서 합니다.

**결과 파일**
- 임베딩: `data/embeddings/{모델}__{문서구성}.npy` (data/는 커밋하지 않음, 다시 실행하면 재사용)
- 비교표: `eval/results/embedding_compare_{문서구성}_{날짜}.md / .csv`

In [1]:
import os
import sys
import time
from datetime import datetime
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # .env, data/ 경로 기준
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import torch

from pipeline.load_catalog import read_catalog
from pipeline.search_docs import PRESETS, build
from scripts.check_embedding import PROBES, TOP_K, ranking_stats

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
EMB_DIR = ROOT / "data" / "embeddings"
RESULTS = ROOT / "eval" / "results"
EMB_DIR.mkdir(parents=True, exist_ok=True)
print("ROOT", ROOT, "| device", DEVICE)

## 1. 설정

- `PRESET`: 검색 문서 구성 (`name` 조사명+표명 / `classes` +분류명 / `full` +분류 값·항목명·경로). 연구계획서 E3 필드 비교는 이 값만 바꿔 다시 실행합니다.
- clir-dolphin 모델은 입력 한도가 약 512토큰이라 `full`에서는 23% 표를 처리하지 못합니다 (`eval/results/embedding_check_20261030.md`).

In [2]:
PRESET = "full"
LOCAL_MODELS = {"bge-m3": "BAAI/bge-m3", "KURE-v1": "nlpai-lab/KURE-v1"}
RUN_CLOVA = False          # True면 CLOVA v2로도 임베딩 (약 40분, 한 번 만들면 파일 재사용)
CLOVA_MODELS = ["v2"]
BATCH_SIZE = 16

## 2. 표 문서 만들기 (카탈로그 파일에서 바로, DB 불필요)

In [3]:
tables = read_catalog()
ids = [t["tbl_id"] for t in tables]
names = [t["table_name"] for t in tables]
docs = [build(t, PRESET) for t in tables]

lengths = pd.Series([len(d) for d in docs])
print(f"표 {len(docs)}개 · 문서 구성 {PRESET}")
print(lengths.describe(percentiles=[0.5, 0.9]).round(0).to_string())
print("\n예시:", docs[ids.index("TX_11025_A020")][:300])

표 2364개 · 문서 구성 full
count    2364.0
mean      281.0
std       106.0
min        65.0
50%       267.0
90%       408.0
max       910.0

예시: 지방자치단체외국인주민현황 | 한국국적을 취득한 자-혼인 귀화자(계)(2007~2015) | 행정구역(시군구)별 국적별 | 서울특별시 부산광역시 대구광역시 인천광역시 광주광역시 대전광역시 울산광역시 세종특별자치시 경기도 강원도 동북아 동남아 남부아 중앙아 미국 러시아 러시아(한국계) 나이지리아 남아프리카공화국 | 혼인 귀화자 | 지방자치단체외국인주민현황 > 2006~2015년(1월1일 기준) > 한국국적을 취득한 자


## 3. 로컬 모델로 임베딩 (파일이 있으면 재사용)

In [4]:
from sentence_transformers import SentenceTransformer


def cache_path(model: str) -> Path:
    return EMB_DIR / f"{model}__{PRESET}.npy"


def load_cache(model: str) -> np.ndarray | None:
    path, id_path = cache_path(model), cache_path(model).with_suffix(".ids")
    if path.exists() and id_path.exists() and id_path.read_text().split() == ids:
        return np.load(path)
    return None


def save_cache(model: str, vectors: np.ndarray) -> None:
    np.save(cache_path(model), vectors)
    cache_path(model).with_suffix(".ids").write_text("\n".join(ids))


vectors: dict[str, np.ndarray] = {}
encoders: dict[str, SentenceTransformer] = {}
timings: dict[str, float] = {}
for label, hf_name in LOCAL_MODELS.items():
    encoders[label] = SentenceTransformer(hf_name, device=DEVICE)
    cached = load_cache(label)
    if cached is not None:
        vectors[label] = cached
        print(f"{label}: 저장된 임베딩 사용 {cached.shape}")
        continue
    start = time.perf_counter()
    vectors[label] = encoders[label].encode(
        docs, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=True
    )
    timings[label] = time.perf_counter() - start
    save_cache(label, vectors[label])
    print(f"{label}: {vectors[label].shape} · {timings[label]:.0f}초")

bge-m3: 저장된 임베딩 사용 (2364, 1024)


KURE-v1: 저장된 임베딩 사용 (2364, 1024)


## 4. CLOVA Studio (선택, `RUN_CLOVA = True`일 때)

In [5]:
from app.services import embedding as clova

for model in CLOVA_MODELS:
    label = f"clova-{model}"
    cached = load_cache(label)
    if cached is not None:
        vectors[label] = cached
        print(f"{label}: 저장된 임베딩 사용 {cached.shape}")
    elif RUN_CLOVA:
        start = time.perf_counter()
        raw = np.array(await clova.embed(docs, model), dtype=np.float32)
        vectors[label] = raw / np.linalg.norm(raw, axis=1, keepdims=True)
        timings[label] = time.perf_counter() - start
        save_cache(label, vectors[label])
        print(f"{label}: {vectors[label].shape} · {timings[label]:.0f}초")
    else:
        print(f"{label}: 건너뜀 (RUN_CLOVA = False, 저장된 임베딩 없음)")

clova-v2: (2364, 1024) · 2366초


## 5. 일상어 질문 10개로 비교

In [6]:
import re

from app.services import search
from app.services import slots as slot_parser

keyword_tables = search.prepare(tables)


async def query_vector(label: str, question: str) -> np.ndarray:
    if label in encoders:
        return encoders[label].encode([question], normalize_embeddings=True)[0]
    raw = np.array((await clova.embed([question], label.removeprefix("clova-")))[0])
    return raw / np.linalg.norm(raw)


rows = []
for question, rule in PROBES:
    relevant = [bool(re.search(rule, n)) for n in names]
    methods = {
        "keyword": search.keyword_scores(
            keyword_tables, search.query_tokens(slot_parser.empty_slots(question))
        )
    }
    for label, matrix in vectors.items():
        methods[label] = (matrix @ await query_vector(label, question)).tolist()
    for method, scores in methods.items():
        stats = ranking_stats(scores, relevant)
        top = sorted(scores, reverse=True)
        top5 = sorted(range(len(scores)), key=lambda i: -scores[i])[:5]
        rows.append({
            "question": question,
            "rule": rule,
            "relevant_tables": sum(relevant),
            "method": method,
            f"hits@{TOP_K}": stats[f"hits@{TOP_K}"],
            "first_rank": stats["first_rank"],
            "score_top1": round(top[0], 3),
            "score_top10": round(top[TOP_K - 1], 3),
            "score_median": round(float(np.median(scores)), 3),
            "top5": " / ".join(names[i] for i in top5),
        })

result = pd.DataFrame(rows)
summary = result.groupby("method", sort=False).agg(
    **{
        f"Top-{TOP_K} 안 관련 표 (평균)": (f"hits@{TOP_K}", "mean"),
        "첫 관련 표가 1위인 질문": ("first_rank", lambda s: int((s == 1).sum())),
        "첫 관련 표 순위 (중앙값)": ("first_rank", "median"),
        "1위 점수 (평균)": ("score_top1", "mean"),
        "중앙값 점수 (평균)": ("score_median", "mean"),
    }
).round(2)
summary

,Top-10 안 관련 표 (평균),첫 관련 표가 1위인 질문,첫 관련 표 순위 (중앙값),1위 점수 (평균),중앙값 점수 (평균)
method,,,,,
keyword,2.0,2,31.0,0.35,0.00
bge-m3,8.5,9,1.0,0.59,0.43
KURE-v1,8.1,9,1.0,0.58,0.41
clova-v2,8.5,9,1.0,0.59,0.43


In [7]:
# 질문별: Top-10 안 관련 표 수 / 첫 관련 표 순위
pivot = result.pivot_table(
    index="question", columns="method", values=[f"hits@{TOP_K}", "first_rank"], sort=False
)
pivot

hits@10                         first_rank         \
method                   keyword bge-m3 KURE-v1 clova-v2    keyword bge-m3   
question                                                                     
노인 인구가 얼마나 늘었는지 볼 수 있는 표     0.0   10.0    10.0     10.0       37.0    1.0   
아기가 몇 명 태어났는지                0.0   10.0     8.0     10.0      108.0    1.0   
결혼 몇 건 했는지                   0.0    7.0     7.0      7.0       43.0    1.0   
이혼한 부부 수                     5.0   10.0    10.0     10.0        1.0    1.0   
혼자 사는 가구                     1.0    6.0     9.0      6.0        3.0    1.0   
우리 동네 외국인 주민 수               7.0   10.0    10.0     10.0        4.0    1.0   
귀화한 사람 통계                    7.0    5.0     0.0      5.0        1.0    5.0   
돌아가신 분 수                     0.0   10.0    10.0     10.0      161.0    1.0   
다른 지역으로 이사 간 사람              0.0   10.0    10.0     10.0       25.0    1.0   
앞으로 인구가 어떻게 될지 예측            0.0    7.0     7.0      7.0     2024.0    1.0   

                                           
method                   KURE-v1 clova-v2  
question                                   
노인 인구가 얼마나 늘었는지 볼 수 있는 표     1.0      1.0  
아기가 몇 명 태어났는지                1.0      1.0  
결혼 몇 건 했는지                   1.0      1.0  
이혼한 부부 수                     1.0      1.0  
혼자 사는 가구                     1.0      1.0  
우리 동네 외국인 주민 수               1.0      1.0  
귀화한 사람 통계                   13.0      5.0  
돌아가신 분 수                     1.0      1.0  
다른 지역으로 이사 간 사람              1.0      1.0  
앞으로 인구가 어떻게 될지 예측            1.0      1.0

## 6. 질문별 상위 5개 표 (정성 확인)

In [8]:
pd.set_option("display.max_colwidth", None)
result[["question", "method", "top5"]]

,question,method,top5
0,노인 인구가 얼마나 늘었는지 볼 수 있는 표,keyword,"시도별 경력단절 여성이 있는 가구(20세이상, 일반가구)-시도 / 점유형태별 경력단절 여성이 있는 가구(20세이상, 일반가구)-시군구 / 시도별 경력단절 여성이 있는 가구(20세이상, 일반가구)-시도 / 점유형태별 경력단절 여성이 있는 가구(20세이상, 일반가구)-시군구 / 영유아가 있는 가구의 자녀수별 가구 및 연령별 영유아 - 시도"
1,노인 인구가 얼마나 늘었는지 볼 수 있는 표,bge-m3,연령/성/혼인상태별 고령자(60세 이상)-시도 / 시도/연령/성/취업상태별(60세 이상) 고령자인구 / 시도/연령/성/세대구성별 고령자인구 / 행정구역/연령/성별 고령자 / 성/연령/가구주와의 관계별 고령자(60세 이상)-시도
2,노인 인구가 얼마나 늘었는지 볼 수 있는 표,KURE-v1,행정구역/성/연령별 고령자 1인가구 / 행정구역/연령/성별 고령자 / 연령/성/혼인상태별 고령자(60세 이상)-시도 / 시도/연령/성/세대구성별 고령자인구 / 행정구역/연령/성별 고령자
3,노인 인구가 얼마나 늘었는지 볼 수 있는 표,clova-v2,연령/성/혼인상태별 고령자(60세 이상)-시도 / 시도/연령/성/취업상태별(60세 이상) 고령자인구 / 시도/연령/성/세대구성별 고령자인구 / 행정구역/연령/성별 고령자 / 성/연령/가구주와의 관계별 고령자(60세 이상)-시도
4,아기가 몇 명 태어났는지,keyword,시도/모의연령/혼인상태별 적생아수 / 시도/모의연령/혼인상태별 적생아수 / 시도/모의연령/혼인상태별 적생아수 / 시도/모의 연령/혼인상태별 적생아 / 시도/모의 연령/혼인상태별 적생아
5,아기가 몇 명 태어났는지,bge-m3,시도/모의 연령/출생아수별 적생아 / 혼인상태별/출생자녀수별/총 출산기간별 기혼여성인구(15세이상)-시도 / 사망자녀수별/출생자녀수별 기혼여성인구(15세이상)-시도 / 성/모의 연령(각세)/출산순위별 출생 / 혼인상태별/출생자녀수별/총 출산기간별 기혼여성인구(15세이상)-시도
6,아기가 몇 명 태어났는지,KURE-v1,성/모의 연령(각세)/출산순위별 출생 / 시도/성/모의 연령(5세계급)/출산순위별 출생 / 시도/임신기간별 출생(쌍태아 이상) / 출생자녀수/초혼연령별 기혼여성인구(15세 이상)-시도 / 시도/모의 연령/출생아수별 적생아
7,아기가 몇 명 태어났는지,clova-v2,시도/모의 연령/출생아수별 적생아 / 혼인상태별/출생자녀수별/총 출산기간별 기혼여성인구(15세이상)-시도 / 성/모의 연령(각세)/출산순위별 출생 / 사망자녀수별/출생자녀수별 기혼여성인구(15세이상)-시도 / 혼인상태별/출생자녀수별/총 출산기간별 기혼여성인구(15세이상)-시도
8,결혼 몇 건 했는지,keyword,"연령별/결혼전 취업여부별/경력단절여부별 여성인구(20세이상, 일반 가구)-시도 / 연령별/결혼전 취업여부별/경력단절여부별 여성인구(20세이상, 일반 가구)-시도 / 시도/모의 연령/결혼생활기간별 출생 / 시도/모의 연령/결혼생활기간별 출생(쌍태아 이상) / 국적(지역)별 결혼이민자 현황"
9,결혼 몇 건 했는지,bge-m3,시도/가구주의 혼인상태/성/가구원수별 개인가구수 / 연령/교육정도별 혼인(초혼) / 연령/교육정도별 혼인 / 연령(5세)/교육정도별 이혼 / 초혼부부의 연령별 혼인


## 7. 결과 저장 (`eval/results/`)

In [9]:
stem = RESULTS / f"embedding_compare_{PRESET}_{datetime.now():%Y%m%d}"
result.to_csv(stem.with_suffix(".csv"), index=False, encoding="utf-8-sig")
lines = [
    f"# 임베딩 모델 비교 ({PRESET}) {datetime.now():%Y-%m-%d %H:%M}",
    "",
    f"- 문서 구성: `{PRESET}` (pipeline/search_docs.py), 표 {len(docs)}개, "
    f"문서 길이 중앙값 {lengths.median():.0f}자",
    f"- 로컬 모델 실행: {DEVICE} · 임베딩 시간(초): "
    + (", ".join(f"{k} {v:.0f}" for k, v in timings.items()) or "저장된 임베딩 재사용"),
    "- 관련 표 = 표명에 규칙 단어가 있는 표. **골든셋이 아닌 규칙 기반 점검**이라 정확도 주장에는 쓰지 않는다.",
    "",
    "## 요약",
    "",
    "| 방식 | " + " | ".join(summary.columns) + " |",
    "|---|" + "---|" * len(summary.columns),
    *[f"| {m} | " + " | ".join(str(v) for v in row) + " |" for m, row in summary.iterrows()],
    "",
    "## 질문별 (Top-10 안 관련 표 수 / 첫 관련 표 순위)",
    "",
]
methods = list(dict.fromkeys(result["method"]))
lines.append("| 질문 | 관련 표 수 | " + " | ".join(methods) + " |")
lines.append("|---|---|" + "---|" * len(methods))
for question, group in result.groupby("question", sort=False):
    g = group.set_index("method")
    cells = [f"{g.loc[m, f'hits@{TOP_K}']} / {g.loc[m, 'first_rank']}" for m in methods]
    lines.append(f"| {question} | {g['relevant_tables'].iloc[0]} | " + " | ".join(cells) + " |")
stem.with_suffix(".md").write_text("\n".join(lines) + "\n", encoding="utf-8")
print("저장:", stem.with_suffix(".md"), stem.with_suffix(".csv"))

저장: eval/results/embedding_compare_full_20261031.md eval/results/embedding_compare_full_20261031.csv
